In [0]:
dbutils.widgets.text("catalog",        "banking")
dbutils.widgets.text("schema_landing", "landing")
dbutils.widgets.text("schema_bronze",  "bronze")

catalog        = dbutils.widgets.get("catalog")
schema_landing = dbutils.widgets.get("schema_landing")
schema_bronze  = dbutils.widgets.get("schema_bronze")
monitoring_table   = f"{catalog}.{schema_bronze}.execution_monitoring"
target_table       = f"{catalog}.{schema_bronze}.bronze_credit"
volume_credit=f'/Volumes/{catalog}/{schema_landing}/credit_bureau_reports'

In [0]:
from pyspark.sql.functions import current_timestamp, lit ,input_file_name

df_credit = spark.read.format("csv") \
    .option("header", "true") \
    .load(volume_credit)

df_credit = df_credit.withColumn('ingestion_date', current_timestamp()) \
                     .withColumn('path', df_credit['_metadata']['file_path'])

df_credit.write.mode("overwrite").option("delta.feature.allowColumnDefaults", "supported").saveAsTable(target_table)